In [ ]:
import pandas as pd

df = pd.read_excel(r"/kaggle/working/datasets-dgtx/DATA_DG.xlsx")   # hoặc path đúng của bạn

assert {"content", "summary", "grade"}.issubset(df.columns)
df = df.dropna().reset_index(drop=True)


from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "VietAI/vit5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer.model_max_length)  # ~512 tokens

from datasets import Dataset

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 256

def preprocess(batch):
    prompts = [
        f"Tóm tắt văn bản cho học sinh lớp {g}: {c}"
        for c, g in zip(batch["content"], batch["grade"])
    ]

    model_inputs = tokenizer(
        prompts,
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LEN,
    )

    labels = tokenizer(
        batch["summary"],
        truncation=True,
        max_length=MAX_TARGET_LEN,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

tokenized_ds = dataset.map(
    preprocess,
    batched=True,   # ⭐ BẮT BUỘC
    remove_columns=dataset["train"].column_names
)

print(tokenized_ds["train"].column_names)

In [ ]:
# ============================================
# CELL 1: OPTUNA HYPERPARAMETER TUNING
# ============================================

import optuna
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = 1
    epochs = trial.suggest_int("num_train_epochs", 3, 8)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    torch.cuda.empty_cache()
    import gc
    gc.collect()

    args = TrainingArguments(
        output_dir=f"./optuna_trial_{trial.number}",
        eval_strategy="no",
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="no",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        label_smoothing_factor=0.1,
        max_grad_norm=1.0,
        fp16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        ddp_find_unused_parameters=False,
        report_to="none"
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=None
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=None,
    )

    trainer.train()
    train_loss = trainer.state.log_history[-1].get("train_loss", float('inf'))
    
    del model
    del trainer
    del data_collator
    del args
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    return train_loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

best_params = study.best_params
print("=" * 50)
print("BEST HYPERPARAMETERS:")
print("=" * 50)
for key, value in best_params.items():
    print(f"{key}: {value}")
print("=" * 50)

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    import numpy as np
    
    # preds từ Trainer là logits, cần lấy argmax để có token IDs
    if isinstance(preds, tuple):
        preds = preds[0]
    
    if isinstance(preds, np.ndarray):
        # Lấy token ID có xác suất cao nhất (argmax)
        if preds.ndim == 3:  # (batch_size, seq_len, vocab_size)
            preds = np.argmax(preds, axis=-1)
        elif preds.ndim > 3:
            preds = preds.reshape(preds.shape[0], -1)
            preds = np.argmax(preds, axis=-1)
    
    # Xử lý labels - loại bỏ padding tokens (-100)
    if isinstance(labels, np.ndarray):
        labels = labels.tolist()
    
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Decode labels - loại bỏ -100 trước khi decode
    decoded_labels = []
    for label in labels:
        if isinstance(label, np.ndarray):
            label = label.tolist()
        # Loại bỏ -100 (ignore index)
        valid_label = [l for l in label if l != -100]
        decoded_labels.append(tokenizer.decode(valid_label, skip_special_tokens=True))

    # Đảm bảo decoded_preds và decoded_labels là list các string
    decoded_preds = [str(p) if p else "" for p in decoded_preds]
    decoded_labels = [str(l) if l else "" for l in decoded_labels]

    # -------- ROUGE --------
    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # -------- BERTScore --------
    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="vi",
        model_type="bert-base-multilingual-cased"
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_precision": sum(bert_result["precision"]) / len(bert_result["precision"]),
        "bertscore_recall": sum(bert_result["recall"]) / len(bert_result["recall"]),
        "bertscore_f1": sum(bert_result["f1"]) / len(bert_result["f1"]),
    }
    
import pandas as pd

def save_metrics(trainer, filename="metrics.csv"):
    logs = trainer.state.log_history
    df_metrics = pd.DataFrame(logs)
    df_metrics.to_csv(filename, index=False)
    return df_metrics

In [ ]:
# ============================================
# CELL 2: FINAL TRAINING VỚI BEST PARAMETERS
# ============================================

from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
import torch
import os

torch.cuda.empty_cache()

# Kiểm tra xem có checkpoint để resume không
resume_from_checkpoint = None
checkpoint_dir = "./vit5_final"
if os.path.exists(checkpoint_dir):
    checkpoints = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint-")]
    if checkpoints:
        # Lấy checkpoint mới nhất
        checkpoint_numbers = [int(c.split("-")[1]) for c in checkpoints]
        latest_checkpoint = f"checkpoint-{max(checkpoint_numbers)}"
        resume_from_checkpoint = os.path.join(checkpoint_dir, latest_checkpoint)
        print(f"Tìm thấy checkpoint: {resume_from_checkpoint}")
        print("Sẽ tiếp tục training từ checkpoint này...")
    else:
        print("Không tìm thấy checkpoint, bắt đầu training từ đầu")
else:
    print("Không tìm thấy thư mục checkpoint, bắt đầu training từ đầu")

best_learning_rate = 0.0003290826760268271
best_num_train_epochs = 5
best_weight_decay = 0.004878586459798251

final_args = TrainingArguments(
    output_dir="./vit5_final",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=best_learning_rate,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=best_num_train_epochs,
    weight_decay=best_weight_decay,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,
    save_total_limit=5,
    report_to="none"
)

# Load model từ checkpoint nếu có, nếu không thì từ pretrained
if resume_from_checkpoint:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(resume_from_checkpoint)
else:
    final_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

final_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=final_model,
    padding=True,
    pad_to_multiple_of=None
)

final_trainer = Trainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=final_data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("=" * 50)
print("BẮT ĐẦU TRAINING VỚI BEST PARAMETERS")
if resume_from_checkpoint:
    print(f"Resume từ checkpoint: {resume_from_checkpoint}")
print("=" * 50)

if resume_from_checkpoint:
    final_trainer.train(resume_from_checkpoint=resume_from_checkpoint)
else:
    final_trainer.train()

final_trainer.save_model("./vit5_grade_summary")
tokenizer.save_pretrained("./vit5_grade_summary")

print("=" * 50)
print("TRAINING HOÀN TẤT!")
print("Model đã được lưu tại: ./vit5_grade_summary")
print("=" * 50)

metrics_df = save_metrics(final_trainer, "vit5_metrics.csv")
print("\nMetrics:")
print(metrics_df.head())
print("\nColumns:", metrics_df.columns.tolist())

import matplotlib.pyplot as plt

eval_df = metrics_df[metrics_df["eval_rouge1"].notna()].copy()
if "epoch" not in eval_df.columns or eval_df["epoch"].isna().all():
    if "step" in eval_df.columns:
        steps_per_epoch = len(tokenized_ds["train"]) // (final_args.per_device_train_batch_size * final_args.gradient_accumulation_steps)
        eval_df["epoch"] = eval_df["step"] / steps_per_epoch
    else:
        eval_df["epoch"] = range(len(eval_df))

if len(eval_df) > 0:
    plt.figure(figsize=(10,6))
    if "eval_rouge1" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rouge1"], label="ROUGE-1")
    if "eval_rougeL" in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df["eval_rougeL"], label="ROUGE-L")
    bertscore_col = "eval_bertscore_f1" if "eval_bertscore_f1" in eval_df.columns else "bertscore_f1"
    if bertscore_col in eval_df.columns:
        plt.plot(eval_df["epoch"], eval_df[bertscore_col], label="BERTScore-F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Evaluation Metrics over Epochs")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Warning: No evaluation data found for plotting")

In [2]:
# ============================================
# CELL: TEST MODEL
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

# Đường dẫn đến model đã train
MODEL_PATH = "../../Model_DG/vit5_grade_summary"
ORIGINAL_MODEL = "VietAI/vit5-base"

# Kiểm tra xem model có tồn tại không
if not os.path.exists(MODEL_PATH):
    print(f"Model không tồn tại tại: {MODEL_PATH}")
    print("Vui lòng train model trước hoặc kiểm tra đường dẫn")
else:
    print(f"Đang load model từ: {MODEL_PATH}")
    
    # Load tokenizer - thử từ checkpoint trước, nếu lỗi thì dùng model gốc
    # (Tokenizer không thay đổi sau training, chỉ model weights thay đổi)
    try:
        print("Đang load tokenizer từ checkpoint...")
        test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
        print("✓ Tokenizer loaded từ checkpoint (slow tokenizer)")
    except Exception as e1:
        try:
            print(f"⚠ Không thể load slow tokenizer: {str(e1)[:100]}")
            print("Đang thử load fast tokenizer...")
            test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
            print("✓ Tokenizer loaded từ checkpoint (fast tokenizer)")
        except Exception as e2:
            print(f"⚠ Không thể load tokenizer từ checkpoint: {str(e2)[:100]}")
            print(f"⚠ File tokenizer.json có thể bị corrupt")
            print(f"Đang load tokenizer từ model gốc {ORIGINAL_MODEL}...")
            print("(Lưu ý: Tokenizer không thay đổi sau training, nên dùng model gốc vẫn đúng)")
            test_tokenizer = AutoTokenizer.from_pretrained(ORIGINAL_MODEL)
            print("✓ Tokenizer loaded từ model gốc")
    
    # Load model weights từ checkpoint (đây là phần quan trọng - model đã được fine-tune)
    print("\nĐang load model weights từ checkpoint...")
    test_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
    print("✓ Model weights loaded từ checkpoint")
    
    # Chuyển model sang GPU nếu có
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_model.to(device)
    test_model.eval()
    
    print(f"Model đã được load vào: {device}")
    print("=" * 50)

def calculate_relevance_score(summary_text, original_text):
    """
    Tính điểm độ liên quan giữa tóm tắt và văn bản gốc dựa trên keyword overlap
    
    Returns:
        float: Điểm từ 0.0 đến 1.0 (càng cao càng liên quan)
    """
    import re
    
    stopwords = {'là', 'của', 'và', 'với', 'cho', 'để', 'trong', 'trên', 'từ', 'đến', 'một', 'các', 'những', 'này', 'đó', 'đã', 'được', 'sẽ', 'có', 'không', 'theo', 'sau', 'trước', 'khi', 'nếu', 'thì', 'mà', 'nhưng', 'hoặc'}
    
    def extract_keywords(text):
        text = re.sub(r'[^\w\s]', ' ', text.lower())
        words = text.split()
        keywords = [w for w in words if len(w) > 2 and w not in stopwords]
        return set(keywords)
    
    summary_keywords = extract_keywords(summary_text)
    original_keywords = extract_keywords(original_text)
    
    if len(original_keywords) == 0:
        return 1.0
    
    intersection = len(summary_keywords & original_keywords)
    union = len(summary_keywords | original_keywords)
    
    if union == 0:
        return 0.0
    
    jaccard = intersection / union
    coverage = intersection / len(original_keywords) if len(original_keywords) > 0 else 0.0
    relevance = (jaccard * 0.4 + coverage * 0.6)
    
    return relevance


def summarize(text, grade, max_new_tokens=None, use_sampling=True, temperature=0.9, debug=False):
    """
    Tóm tắt văn bản cho học sinh theo lớp
    
    Args:
        text: Văn bản cần tóm tắt
        grade: Lớp học (1-5)
        max_new_tokens: Số token tối đa (nếu None, tự động tính dựa trên 50% số câu)
        use_sampling: Không dùng nữa (luôn dùng sampling để tạo đa dạng)
        temperature: Độ đa dạng của output (0.1-1.5)
            - 0.7-0.9: Đa dạng vừa phải, cân bằng giữa chất lượng và đa dạng (khuyến nghị)
            - 0.5-0.7: Ít đa dạng hơn, ổn định hơn
            - 0.9-1.2: Đa dạng cao, mỗi lần chạy khác nhau nhiều
            - >1.2: Rất đa dạng nhưng có thể kém chính xác
        debug: Nếu True, in thông tin debug
    
    Returns:
        Tóm tắt văn bản (đã được cắt ở câu hoàn chỉnh và loại bỏ phần không liên quan)
        Mỗi lần chạy sẽ tạo ra tóm tắt khác nhau do sử dụng sampling
    """
    import re
    
    if grade < 1:
        grade = 1
    if grade > 5:
        grade = 5
    
    # Tính số câu trong văn bản gốc
    original_sentences = re.split(r'[.!?]+\s*', text)
    original_sentences = [s.strip() for s in original_sentences if s.strip()]
    num_original_sentences = len(original_sentences)
    
    # Tính max_new_tokens dựa trên 50% số câu
    if max_new_tokens is None:
        avg_tokens_per_sentence = {
            1: 15,
            2: 18,
            3: 22,
            4: 25,
            5: 30  # Tăng lên để model generate dài hơn
        }.get(grade, 25)
        
        target_sentences = max(1, int(num_original_sentences * 0.5))
        max_new_tokens = target_sentences * avg_tokens_per_sentence
        
        # Tăng giới hạn đáng kể để ép model generate dài hơn (vì model được train với MAX_TARGET_LEN=64)
        # Với 21 câu gốc, 50% = ~10-11 câu, mỗi câu 30 tokens = 300-330 tokens
        min_limit = {1: 120, 2: 150, 3: 200, 4: 250, 5: 350}.get(grade, 300)
        max_limit = {1: 300, 2: 400, 3: 500, 4: 650, 5: 800}.get(grade, 700)
        max_new_tokens = max(min_limit, min(max_new_tokens, max_limit))
    
    # Tăng min_new_tokens lên 50% để đảm bảo tóm tắt đủ dài
    min_new_tokens = int(max_new_tokens * 0.5)
    
    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {text}"
    
    inputs = test_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        # Dùng sampling với temperature để tạo sự đa dạng mỗi lần chạy
        # Temperature cao hơn = đa dạng hơn, nhưng có thể kém chính xác hơn
        output = test_model.generate(
            **inputs,
            min_new_tokens=min_new_tokens,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # Bật sampling để tạo đa dạng
            temperature=temperature,  # Temperature từ parameter (mặc định 0.9)
            top_p=0.92,  # Nucleus sampling - chỉ xét top 92% tokens
            top_k=50,  # Top-k sampling - chỉ xét top 50 tokens
            length_penalty=2.0,  # Khuyến khích generate dài
            repetition_penalty=1.2,  # Giảm lặp lại
            no_repeat_ngram_size=2,  # Tránh lặp cụm từ
            eos_token_id=test_tokenizer.eos_token_id,
            pad_token_id=test_tokenizer.pad_token_id
        )
    
    summary_raw = test_tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Debug: Tính toán thông tin về summary gốc
    original_summary_length = len(summary_raw.split())
    # Tính số tokens đã generate (loại bỏ input tokens)
    input_length = inputs['input_ids'].shape[1]
    output_length = output[0].shape[0]
    num_generated_tokens = output_length - input_length
    num_original_sentences_in_summary = len(re.findall(r'[.!?]', summary_raw))
    
    summary = summary_raw
    
    # Post-processing: Cắt ở câu hoàn chỉnh và loại bỏ phần không liên quan
    sentences = re.split(r'([.!?]+\s*)', summary)
    
    complete_sentences = []
    for i in range(0, len(sentences) - 1, 2):
        if i + 1 < len(sentences):
            sentence = (sentences[i] + sentences[i + 1]).strip()
            if sentence:
                complete_sentences.append(sentence)
    
    # Xử lý câu cuối - CHỈ giữ nếu có dấu kết thúc câu (để tránh cắt giữa câu)
    if len(sentences) % 2 == 1 and sentences[-1].strip():
        last_sentence = sentences[-1].strip()
        # CHỈ giữ nếu có dấu kết thúc câu (hoàn chỉnh)
        if re.search(r'[.!?]$', last_sentence):
            complete_sentences.append(last_sentence)
        # Nếu không có dấu kết thúc, bỏ qua (câu bị cắt)
    
    # Loại bỏ câu không liên quan hoặc lặp lại
    filtered_sentences = []
    seen_content = set()
    min_relevance_threshold = 0.05  # Giảm xuống để giữ nhiều câu hơn
    
    for sentence in complete_sentences:
        sentence_relevance = calculate_relevance_score(sentence, text)
        
        # Chỉ lọc câu có độ liên quan RẤT thấp (< 0.03) - rất lỏng để giữ nhiều nội dung
        if sentence_relevance < 0.03:
            continue
        
        normalized = re.sub(r'\s+', ' ', sentence.lower().strip())
        
        # Kiểm tra duplicate (tăng ngưỡng lên 0.9 để linh hoạt hơn, chỉ lọc câu gần như giống hệt)
        is_duplicate = False
        for seen in seen_content:
            if len(normalized) > 0 and len(seen) > 0:
                similarity = len(set(normalized.split()) & set(seen.split())) / max(len(normalized.split()), len(seen.split()))
                if similarity > 0.9:  # Tăng lên 0.9 để chỉ lọc câu gần như giống hệt
                    is_duplicate = True
                    break
        
        if not is_duplicate:
            filtered_sentences.append(sentence)
            seen_content.add(normalized)
    
    # Xử lý kết quả cuối cùng - GIỮ TẤT CẢ các câu hợp lệ, không dừng sớm
    if filtered_sentences:
        # Chỉ loại bỏ câu có độ liên quan RẤT thấp ở cuối (nếu có nhiều câu)
        # Nhưng vẫn giữ tất cả các câu hợp lệ
        final_summary = []
        for i, sentence in enumerate(filtered_sentences):
            sentence_relevance = calculate_relevance_score(sentence, text)
            
            # Chỉ bỏ qua câu có độ liên quan RẤT thấp (< 0.03) và chỉ khi đã có ít nhất 5 câu
            # Điều này đảm bảo giữ được nhiều nội dung
            if sentence_relevance < 0.03 and len(final_summary) >= 5:
                # Kiểm tra xem câu tiếp theo có tốt hơn không
                if i + 1 < len(filtered_sentences):
                    next_relevance = calculate_relevance_score(filtered_sentences[i+1], text)
                    if next_relevance < 0.03:
                        # Nếu câu tiếp theo cũng rất thấp, dừng ở đây
                        break
                else:
                    # Nếu đây là câu cuối, vẫn giữ nếu đã có ít nhất 5 câu
                    break
            
            final_summary.append(sentence)
        
        summary = ' '.join(final_summary)
    else:
        # Fallback: lấy các câu hoàn chỉnh từ summary gốc (có dấu kết thúc)
        sentences_original = re.split(r'([.!?]+\s*)', summary)
        fallback_sentences = []
        for i in range(0, len(sentences_original) - 1, 2):
            if i + 1 < len(sentences_original):
                sentence = (sentences_original[i] + sentences_original[i + 1]).strip()
                if sentence:
                    fallback_sentences.append(sentence)
        
        if fallback_sentences:
            summary = ' '.join(fallback_sentences)
        else:
            summary = summary.strip()
    
    # Debug output
    if debug:
        print(f"\n[DEBUG] Số câu gốc: {num_original_sentences}")
        print(f"[DEBUG] max_new_tokens: {max_new_tokens}, min_new_tokens: {min_new_tokens}")
        print(f"[DEBUG] Số tokens đã generate: {num_generated_tokens}")
        print(f"[DEBUG] Số từ trong summary gốc: {original_summary_length}")
        print(f"[DEBUG] Số câu trong summary gốc: {num_original_sentences_in_summary}")
        print(f"[DEBUG] Số câu sau post-processing: {len(final_summary) if filtered_sentences else len(fallback_sentences) if fallback_sentences else 0}")
    
    return summary

# ============================================
# TEST VỚI CÁC VÍ DỤ
# ============================================

print("\n" + "=" * 50)
print("TEST MODEL")
print("=" * 50)
test_text_3 = """TÌNH HÌNH XÃ HỘI GIAO CHÂU
Về mặt xã hội, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét cúa bọn quan lại đô hộ đã nhanh chóng đẩy người dân Giao Châu vào con đường bần cùng hóa. 
Sự phân hóa giai cấp trong xã hội ngày càng sâu sắc. Đặc biệt, từ nửa sau thế kỳ VIII, lụt lội và hạn hán xảy ra thường xuyên, chiến tranh liên tiếp do các nước láng giềng là Hoàn 
Vương (Champa) và Nam Chiếu gây ra, khiến sức sản xuất bị phá hoại, đời sống nhân dân ngày càng thêm cơ cực. về mặt văn hoá, nhà Đường đă đu nhập đạo Nho, đạo Phật và đạo Giáo với 
mục đích dựa vào sự phát triển văn hóa để nô dịch nhân dân Giao Châu. Nho giáo thời Đường ở Giao Châu chưa thể xem là phát triển, song cũng được truyền bá sâu rộng trong tầng lớp 
trên của xã hội. Trường học dạy chữ Hán được mờ nhiều ờ các phủ, châu. Trong tầng lớp Hào trưởng người Việt, một số gia đình đã cho con em học hành. Họ được tham gia thi cử và đỗ 
đạt ở Bắc triều, một số người được tuyển dụng vào bộ máy của chính quyền đô hộ, như trường hợp anh em Khương Công Phụ và Khương Công Phục. Tuy nhiên, việc học và thi cử ở Giao Châu 
vẫn bị hạn chế. Lệ nhà Đường năm Hội Xương thứ 5 (845) quy định: "An Nam đưa vào thi Tiến sĩ không được quá tám người, Minh kinh không được quá mười người". Dưới triều Đường, đạo 
Phật được truyền bá vào Giao Châu với hai phái Thiền tông của Phật giáo Trung Quốc. Phái thứ nhất do Tì Ni Đa Lưu Chi sáng lập, truyền bá vào Giao Châu cuối thế kỷ VI, trung tâm là 
chùa Pháp Vân (Thuận Thành, Bắc Ninh). Phái thứ hai do Vô Ngôn Thông sáng lập, truyền vào Giao Châu đầu thế kỷ IX, trung tâm là chùa Kiến Sơ (Phù Đổng, ngoại thành Hà Nội). Bên 
cạnh đạo Phật, Đạo giáo cũng khá phát triển và chịu ảnh hưởng sâu sắc của Trung Quốc. Nhà Đường đã cho nhiều đạo sĩ, phù thủy sang Giao Châu, trong đó có Tiết độ sứ Cao Biền với 
những phương thức tà ma, bùa chú để yểm trừ "long mạch"... Tuy nhiên, sự phát triển của đạo Nho, đạo Phật và Đạo giáo đều dung hòa được với tín ngưỡng dân gian cổ truyền của người 
Việt (như tục thờ thần sông, thần núi, thờ các vị anh hùng dân tộc.. tạo nên sức mạnh của dân tộc Việt, chống lại sự nô dịch văn hoá của ngoại bang."""
test_grade_3 = 5

print(f"\n" + "-" * 50)
print(f"[TEST 3] Lớp {test_grade_3}")
print(f"Văn bản gốc: {test_text_3}")
print(f"\nTóm tắt:")
summary_3 = summarize(test_text_3, test_grade_3, debug=True)
print(summary_3)

print("\n" + "=" * 50)
print("HOÀN TẤT TEST!")
print("=" * 50)

Đang load model từ: ../../Model_DG/vit5_grade_summary
Đang load tokenizer từ checkpoint...
✓ Tokenizer loaded từ checkpoint (slow tokenizer)

Đang load model weights từ checkpoint...
✓ Model weights loaded từ checkpoint
Model đã được load vào: cuda

TEST MODEL

--------------------------------------------------
[TEST 3] Lớp 5
Văn bản gốc: TÌNH HÌNH XÃ HỘI GIAO CHÂU
Về mặt xã hội, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét cúa bọn quan lại đô hộ đã nhanh chóng đẩy người dân Giao Châu vào con đường bần cùng hóa. 
Sự phân hóa giai cấp trong xã hội ngày càng sâu sắc. Đặc biệt, từ nửa sau thế kỳ VIII, lụt lội và hạn hán xảy ra thường xuyên, chiến tranh liên tiếp do các nước láng giềng là Hoàn 
Vương (Champa) và Nam Chiếu gây ra, khiến sức sản xuất bị phá hoại, đời sống nhân dân ngày càng thêm cơ cực. về mặt văn hoá, nhà Đường đă đu nhập đạo Nho, đạo Phật và đạo Giáo với 
mục đích dựa vào sự phát triển văn hóa để nô dịch nhân dân Giao Châu. Nho giáo thời Đường 

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `num_beams` is set to 1. However, `length_penalty` is set to `2.0` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `length_penalty`.
  warnings.warn(



[DEBUG] Số câu gốc: 17
[DEBUG] max_new_tokens: 350, min_new_tokens: 175
[DEBUG] Số tokens đã generate: -230
[DEBUG] Số từ trong summary gốc: 258
[DEBUG] Số câu trong summary gốc: 7
[DEBUG] Số câu sau post-processing: 7
Ngày nay, chính sách bóc lột nặng nề của triều đình nhà Đường và sự tham lam vơ vét đã đẩy người dân Giao Châu vào con đường bần cùng. Từ nửa sau thế kỳ VIII, lụt lội và hạn hán kéo dài làm đời sống nhân dân càng thêm khó khăn, đặc biệt từ cuối thời kỳ VII đến cuối Thế kỳ IX, úng ngập xảy ra thường xuyên khiến sản xuất bị phá hủy hoặc cuộc chiến tiếp diễn. Nho giáo được truyền bá qua hai phái Thiền tông, với hai học trò là Tì Ni Đa Lưu Chi và Vô Ngôn Thông. Đạo Phật cũng có ảnh hưởng tích cực của Trung Quốc, như việc cho đạo sĩ thi cử và đỗ đạt ở Bắc triều. Tuy nhiên, việc học tập ở Giao Chau vẫn bị hạn chế khi Lệ nhà Đàng quy định rằng an Nam đưa tiến sĩ không quá tám người và Minh kinh chỉ khoảng mười người. Khổng giáo còn phát triển khá sâu sắc trong tầng lớp Hào trư

In [5]:
# ============================================
# LOAD MODEL ĐÃ TRAIN
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "../../Model_DG/vit5_grade_summary"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# ============================================
# HÀM TÓM TẮT 1 VĂN BẢN
# ============================================

def summarize_text(content, grade,
                   max_input_len=512,
                   max_target_len=256,
                   mode="sample",
                   length_option="medium"):  # beam | sample

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to(device)

    with torch.no_grad():

        if mode == "beam":
            output_ids = model.generate(
                **inputs,
                max_length=max_target_len,
                num_beams=5,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        elif mode == "sample":
            output_ids = model.generate(
                **inputs,
                max_length=max_target_len,
                do_sample=True,
                top_k=50,
                top_p=0.9,
                temperature=0.8,
                no_repeat_ngram_size=3
            )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary


# ============================================
# TEST VỚI 1 VĂN BẢN BẤT KỲ
# ============================================

text = """
Các già làng thường chỉ tay ra con suối trước nhà chảy từ rừng sâu đổ về thung lũng uốn lượn quanh bản trông giống như một con rắn lớn, để bắt đầu câu chuyện về con nước hiền, con nước dữ. Chỉ nghe một lần là người ta nhớ mãi không quên…
Ở một nơi kia, mãi chỗ tận cùng của con suối có một cái hố thông xuống ruột đất to lắm, sâu lắm. To và sâu đến nỗi nước khắp trời hàng năm rơi xuống, đổ về, không bao giờ đầy. Bên cạnh miệng hố mỗi năm mọc thêm một cây bầu thần kỳ. Hàng năm nõ chỉ đậu một quả. Quả to rất nhanh. Người ta bảo cây bầu ấy do Be Dòng trồng, quả nó che lấp mất lỗ thông xuống ruột đất, nước sẽ ứ ngập khắp trần gian, giết hại loài người và muôn thú.
Thường thì năm nào cũng vậy, khi quả bầu lớn lên chắn ngang miệng hố, nước trên trời rơi xuống không chỗ thoát, tràn khắp đồi thấp, đồi cao. Muôn loài rơi vào thảm họa của nạn hồng thủy. Nhưng rồi quả bầu ấy không hiểu vì sao lại tự vỡ ra. Nước lại chui hết xuống ruột đất. Vào một năm, quả bầu nằm im ở miệng hố trong nhiều lần trăng mọc, nhiều lần trăng chết. Thế là muôn loài chim đắm trong biển nước mênh mông. Người và muông loài muôn thú chết gần hết. Chỉ có chim gò sóng nhờ có đôi cánh khỏe, bay được cao, nên mới qua khỏi cơn biến nạn. Bụng Gò Sóng căm Be Dòng nhiều. Gò Sóng ngày ngày bay tới cái hố thông với ruột đất đó, dùng những chiếc lông cánh vừa cứng vừa nhọn hơn mũi tên của mình, đâm xiên cho vỡ quả bầu. Quả bầu chỉ vỡ từng mảnh nhỏ. Gò Sóng vẫn không nản lòng, hết ngày này đến ngày khác. Đến khi quả bầu vỡ vụn hết mới thôi. Nước miệng hố lại chảy thông với ruột đất.
Sau lần ngập nước kéo dài năm đó, loài người chỉ còn sống sót lại hai anh em Mó Hù và Mó Po. Mó Hù là con trai, Mó Po là con gái. Thấy núi rừng xác xơ trơ trọi. Bụng Mó Hù và Mó Po không vơi nỗi lo âu buồn bã. Ông trời động lòng, sai người xuống trần gian bảo hai anh em:
– Hai người này khóc nhiều, nước mắt khô rồi. Cạn dòng nước mắt cũng đến thế thôi. Bây giờ chúng mày phải thành vợ thành chồng để gây lại nòi giống thôi! Trời bảo thế.
Nghe nói, đầu Mó Hù lắc lắc, mắt Mó Hù nghoảnh đi chớp chớp không dám nhìn Mó Po. Mãi sau Mó Hù mới thở dài nơi với người trời:
– Không được đâu người trời à! Mó Po là em gái tôi, ngủ chung làm vợ làm chồng sao được! Bụng mình không ưng đâu!
Biết rằng lời nói suông khó có thể thay đổi được ý nghĩ của loài người, người trời lựa lời nói:
– Ta sẽ cho chúng mày hai hạt giống. Mỗi người trồng một hạt xa nhau mười lần cánh tay. Nếu hai cây mọc lên mà tự tìm về gốc quấn lấy nhau thì chúng mày phải nên vợ nên chồng thôi! Ý trời bảo thế đấy!
Mó Hù và Mó Po nhận hai hạt giống đem trồng như lời người trời nói. Sáng hôm sau cả hai hạt giống đã nảy mần. Hạt Mó Hù trông mọc thành dây leo, hạt Mó Po trồng đâm lên cây thẳng. Rất lạn, cả hai cây lớn rất nhanh. Sang ngày hôm sau rồi hôm sau nữa, dây của cây Mó Hù đã bò tới quấn lên thân cây. Rồi chúng bắt đầu đâm hoa kết trái.
Thấy vật Mó Hù và Mó Po cùng nghĩ: Trời đã định thế ta phải làm theo thôi! Trái ý trời sao được. Thế là Mó Hù và Mó Po nên vợ nên chồng. Rồi họ sinh con, nhiều trai nhiều gái. Con  họ lại sinh thêm nhiều cháu. Loài người từ đó ngày một thêm đông đúc. Từng bản, từng bản lại được dựng lên. Hai vợ chồng Mó Hù chờ cho cây đó ra quả, lại lấy hạt gieo khắp núi cao, đồi thấp, thung xa. Rồi cây cối lại sinh sôi mọc ngút ngàn. Kết thành từng rừng lớn. rừng nhỏ. Còn loài chim Gò Sóng vẫn sinh sôi này nở. Các loài chim thú khác cũng ngày một sinh sôi. Từ đó, để cho cuộc sống loài người cũng nhe muôn loài được yên ổn, Gò Sóng vẫn ngày ngày bay đến chỗ có lỗ nước thông với ruột đất đẻ xem nếu có nguy cơ của nạn hông thủy như năm nọ thì chim lại dùng những chiếc lông cánh của mình bắn vỡ quả bầu thần. Năm nào Gò Sóng mải kiếm ăn, quên đi bắn thì quả bầu lớn nhanh chắn ngang miệng hố và nạn lụt lan lan tràn…Khi ấy người dân ở đây thường bảo nhau cầu khấn chim Gò Sóng mau trở lại để phá quả bầu thần, cứu cho muôn loài khỏi gập lụt.
Chuyện về con nước hiền, con nước dữ ngừng ở đây, thường thì như vậy. Nhưng nếu câu chuyện được kể vào tháng sáu, tháng bảy khi những cơn giông gầm thét ầm ào ngoài rừng, từng đợt mưa trút xuống thì các già làng lại kể đoạn tiếp đoạn sau này nữa:
…Thấy chưa! Gò Sóng và Be Dòng đang đánh nhau đấy!
Gò sóng dùng cánh bắn Be Dòng tạo nên gió lớn. Còn Be Dòng dâng nước lên cao để định cuốn gò sóng xuống làm cho mưa to. Sau những trận giao tranh quyết liệt, Gò Sóng thường bắn hết lông cánh. Nó phải mượn đũa của con người để bắn tiếp. Vì vậy vào tháng sáu, tháng bảy con người bị mất đũa thường ít kêu ca, tìm kiếm. Còn khi vào rừng thấy Gò Sóng gãy cánh hay rụng hết lông rớt xuống, người ta mang về nuôi chu đáo. Đến khi Gò Sóng đủ lông liền cánh lại thả nó về rừng. Con người không nỡ giết thịt loài chim đã giúp mình thoát khỏi những nạn lụt khủng khiếp, chính là từ câu chuyện này."""

grade = 5

result = summarize_text(text, grade)
print("Văn bản: ", text)
print("Tóm tắt: ", result)

Văn bản:  
Các già làng thường chỉ tay ra con suối trước nhà chảy từ rừng sâu đổ về thung lũng uốn lượn quanh bản trông giống như một con rắn lớn, để bắt đầu câu chuyện về con nước hiền, con nước dữ. Chỉ nghe một lần là người ta nhớ mãi không quên…
Ở một nơi kia, mãi chỗ tận cùng của con suối có một cái hố thông xuống ruột đất to lắm, sâu lắm. To và sâu đến nỗi nước khắp trời hàng năm rơi xuống, đổ về, không bao giờ đầy. Bên cạnh miệng hố mỗi năm mọc thêm một cây bầu thần kỳ. Hàng năm nõ chỉ đậu một quả. Quả to rất nhanh. Người ta bảo cây bầu ấy do Be Dòng trồng, quả nó che lấp mất lỗ thông xuống ruột đất, nước sẽ ứ ngập khắp trần gian, giết hại loài người và muôn thú.
Thường thì năm nào cũng vậy, khi quả bầu lớn lên chắn ngang miệng hố, nước trên trời rơi xuống không chỗ thoát, tràn khắp đồi thấp, đồi cao. Muôn loài rơi vào thảm họa của nạn hồng thủy. Nhưng rồi quả bầu ấy không hiểu vì sao lại tự vỡ ra. Nước lại chui hết xuống ruột đất. Vào một năm, quả bầu nằm im ở miệng hố trong nhi

Níc Vu-gic, một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân với hai ngón chân nhỏ. Do khác biệt về ngoại hình, Níc đã học cách dùng chân và một cái cán để viết chữ, đánh bàn phím máy vi tính và chăm sóc bản thân. Anh đã vượt qua khó khăn để biến bản thân thành điều kì diệu trong cuộc sống. Cuối cùng, Ní đã biến bản Thân trở thành một biểu tượng của nghị lực sống và lan toả năng lượng tích cực tới người xung quanh.

Níc Vu-gic là một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân nhỏ. Sự khác biệt về ngoại hình khiến anh từng bị bạn bè chê cười. Trong thời gian khó khăn, Níc phải chấp nhận sống chung với sự thiếu sót trên cơ thể mình và học cách dùng chân và cán để viết chữ và đánh bàn phím. Anh đã vượt qua khó khăn để biến mình thành điều kì diệu trong cuộc sống.

Níc Vu-gic là một diễn giả truyền cảm hứng người Úc, sinh năm 1982 và chỉ có một bàn chân với hai ngón chân nhỏ. Sự khác biệt về ngoại hình khiến Níc từng bị bạn bè chê cười và trêu chọc. Trong thời gian khó khăn, Níc đã học cách dùng chân và một cái cán để viết chữ, đánh bàn phím máy vi tính và chăm sóc bản thân. Anh đã vượt qua khó khăn và biến bản thân thành điều kì diệu trong cuộc sống.

In [23]:
# ============================================
# LOAD MODEL
# ============================================
import re
import unicodedata
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from vncorenlp import VnCoreNLP

MODEL_PATH = "../../Model_DG/vit5_grade_summary"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

model.to(device)
model.eval()


# ============================================
# LOAD VNCORENLP
# ============================================

vncore = VnCoreNLP(
    "../../../VnCoreNLP-master/VnCoreNLP-1.1.1.jar",
    annotators="wseg,pos,ner",
    max_heap_size='-Xmx2g'
)


# ============================================
# ĐẾM SỐ TỪ
# ============================================

def count_words(text):

    sentences = vncore.tokenize(text)

    words = []
    for sent in sentences:
        words.extend(sent)

    return len(words)


# ============================================
# WORD → TOKEN CONVERSION
# ============================================

def word_to_token_estimate(word_count):
    """
    Với tiếng Việt:
    1 word ≈ 1.3 token (thực nghiệm với tokenizer T5)
    """
    return int(word_count * 1.3)


# ============================================
# CẮT SUMMARY Ở CÂU HOÀN CHỈNH
# ============================================

def clean_summary(summary):

    last_period = summary.rfind(".")

    if last_period != -1:
        summary = summary[:last_period + 1]

    return summary.strip()


def clean_vietnamese_text(text):
    
    # 1 normalize unicode
    text = unicodedata.normalize("NFC", text)

    # 2 fix double spaces
    text = re.sub(r"\s+", " ", text)

    # 3 fix dính chữ hoa
    text = re.sub(r"([a-zàáạảãăắằẳẵặâấầẩẫậ])([A-Z])", r"\1 \2", text)

    # 4 fix chữ viết hoa toàn bộ
    def fix_upper(word):
        if word.isupper() and len(word) > 2:
            return word.capitalize()
        return word

    text = " ".join([fix_upper(w) for w in text.split()])

    # 5 fix các lỗi phổ biến
    corrections = {
        "Đĩnhchi": "Đĩnh Chi",
        "Mạc Đĩnhchi": "Mạc Đĩnh Chi",
        "Đĩnh chi": "Đĩnh Chi",
        "Đĩnh CHI": "Đĩnh Chi",
    }

    for wrong, correct in corrections.items():
        text = text.replace(wrong, correct)

    return text

import re
import unicodedata


def vietnamese_text_normalization(text, vncore=None):
    """
    Vietnamese Text Normalization Layer
    Works for ANY Vietnamese text
    """

    # ====================================
    # 1. Unicode normalization
    # ====================================
    text = unicodedata.normalize("NFC", text)

    # ====================================
    # 2. Remove duplicated spaces
    # ====================================
    text = re.sub(r"\s+", " ", text)

    # ====================================
    # 3. Fix spacing around punctuation
    # ====================================
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"([,.!?;:])([^\s])", r"\1 \2", text)

    # ====================================
    # 4. Fix glued words
    # Example: Đĩnhchi -> Đĩnh Chi
    # ====================================
    text = re.sub(
        r"([a-zàáạảãăắằẳẵặâấầẩẫậđèéẹẻẽêếềểễệìíịỉĩòóọỏõôốồổỗộơớờởỡợùúụủũưứừửữựỳýỵỷỹ])([A-ZÀÁẠẢÃĂẮẰẲẴẶÂẤẦẨẪẬĐ])",
        r"\1 \2",
        text
    )

    # ====================================
    # 5. Fix ALL CAPS words
    # ====================================
    words = text.split()

    fixed_words = []
    for w in words:
        if w.isupper() and len(w) > 2:
            fixed_words.append(w.capitalize())
        else:
            fixed_words.append(w)

    text = " ".join(fixed_words)

    # ====================================
    # 6. Capitalize sentences
    # ====================================
    sentences = re.split(r'(?<=[.!?]) +', text)

    def capitalize_first_letter(sentence):
        if not sentence:
            return sentence
        return sentence[0].upper() + sentence[1:]

    sentences = [capitalize_first_letter(s) for s in sentences]

    text = " ".join(sentences)

    # ====================================
    # 7. Use NER to fix proper names
    # ====================================
    if vncore is not None:

        annotated = vncore.annotate(text)

        entities = set()

        for sent in annotated['sentences']:
            for token in sent:
                if token['nerLabel'] != 'O':
                    entities.add(token['form'])

        for ent in entities:
            pattern = re.compile(ent, re.IGNORECASE)
            text = pattern.sub(ent, text)

    return text.strip()

# ============================================
# HÀM TÓM TẮT
# ============================================

def summarize_text(
        content,
        grade,
        length_option="medium",
        max_input_len=512,
        mode="sample"
):

    # -------------------------------
    # 1. ĐẾM SỐ TỪ VĂN BẢN
    # -------------------------------

    total_words = count_words(content)

    # -------------------------------
    # 2. TÍNH ĐỘ DÀI SUMMARY
    # -------------------------------

    if length_option == "short":
        summary_words = int(total_words * 0.35)

    elif length_option == "medium":
        summary_words = int(total_words * 0.50)

    elif length_option == "long":
        summary_words = int(total_words * 0.75)

    else:
        summary_words = int(total_words * 0.50)

    # -------------------------------
    # 3. WORD → TOKEN
    # -------------------------------

    max_len = word_to_token_estimate(summary_words)
    min_len = int(max_len * 0.6)

    # tránh quá dài
    max_len = min(max_len, 256)

    # -------------------------------
    # 4. TOKENIZE INPUT
    # -------------------------------

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to(device)

    # -------------------------------
    # 5. GENERATE
    # -------------------------------

    with torch.no_grad():

        if mode == "beam":

            output_ids = model.generate(
                **inputs,
                min_length=min_len,
                max_length=max_len,
                num_beams=5,
                length_penalty=1.0,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        else:

            output_ids = model.generate(
                **inputs,
                min_length=min_len,
                max_length=max_len,
                do_sample=True,
                top_k=50,
                top_p=0.9,
                temperature=0.8
            )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )
    summary = vietnamese_text_normalization(summary, vncore)
    summary = clean_summary(summary)

    return summary, total_words, summary_words


# ============================================
# TEST
# ============================================

text = """Nhà họ Mạc ở Lũng Động sinh được một người con trai, đặt tên là Mạc Đĩnh Chi. 
Cậu bé càng lớn càng thông minh, lại rất chăm chỉ học hành và có tài ứng đối mau lẹ. 
Năm đó, Mạc Đĩnh Chi về kinh dự thi và đỗ đầu. Khi vào chầu, vua thấy dung mạo của ông 
không đẹp nên muốn thử tài một lần nữa. Nhà vua ướm hỏi ông về những điều cần có của 
một người thi đỗ. Mạc Đĩnh Chi tâu vua xin được trả lời bằng giấy bút. Giây lát sau, 
ông dâng vua một bài phú có nhan đề 'Bông sen trong giếng ngọc' để tỏ rõ chỉ hướng và 
tài năng của mình. Chữ Mạc Đĩnh Chi rất đẹp, bài phú lại hay, phô bày vẻ đẹp, hương thơm 
của bông sen trong giếng nước. Nhờ bông hoa mà giếng trở thành giếng quý. Xem xong bài phú, 
vua Trần Anh Tông quyết định chọn Mạc Đĩnh Chi làm trạng nguyên của khoa thi ấy. 
Với lòng yêu nước, thương dân, Mạc Đĩnh Chi đã làm được nhiều việc lớn cho đất nước."""

grade = 5

for mode in ["short", "medium", "long"]:

    summary, total_words, sum_words = summarize_text(
        text,
        grade,
        length_option=mode
    )

    print("\n=======================")
    print("Mode:", mode)
    print("Số từ văn bản:", total_words)
    print("Số từ summary target:", sum_words)
    print("Summary:", summary)


Mode: short
Số từ văn bản: 185
Số từ summary target: 64
Summary: Nhà họ Mạc ở Lũng Động có một người con trai tên là Mạc Đĩnh Chi, càng lớn càng thông minh và chăm chỉ học hành. Khi về kinh dự thi, Mạc Đĩnh Chi đỗ đầu và được vua hỏi về những điều cần có của một người thi đỗ. Nhà vua rất muốn thử tài cậu và muốn thử tài ông.

Mode: medium
Số từ văn bản: 185
Số từ summary target: 92
Summary: Nhà họ Mạc ở Lũng Động có một cậu bé tên là Mạc Đĩnh Chi, thông minh, chăm chỉ học hành và có tài ứng đối nhanh. Khi về kinh dự thi, vua thấy cậu không đẹp nên muốn thử tài. Mạc Đĩnh Chi tâu vua và xin được trả lời bằng giấy bút. Sau đó, Mạc Đĩnh Chi dâng vua một bài phú có tên 'Bông sen trong giếng ngọc', thể hiện tài năng của cậu. Chữ của cậu rất đẹp và dễ hiểu, phô bày vẻ đẹp của bông sen trong giếng nước.

Mode: long
Số từ văn bản: 185
Số từ summary target: 138
Summary: Nhà họ Mạc ở Lũng Động có một cậu bé tên là Mạc Đĩnh Chi, nhỏ nhắn và chăm chỉ học hành. Khi về kinh dự thi, vua thấy cậu khôn

In [3]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from underthesea import word_tokenize
import torch

# ---------------------------
# Tiền xử lý tiếng Việt
# ---------------------------
def preprocess_vi(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens


# ---------------------------
# Tính ROUGE
# ---------------------------
def compute_rouge(candidate, reference):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=False
    )

    scores = scorer.score(reference, candidate)

    rouge_results = {
        "rouge1_precision": scores['rouge1'].precision,
        "rouge1_recall": scores['rouge1'].recall,
        "rouge1_f1": scores['rouge1'].fmeasure,
        "rougeL_precision": scores['rougeL'].precision,
        "rougeL_recall": scores['rougeL'].recall,
        "rougeL_f1": scores['rougeL'].fmeasure,
    }

    return rouge_results


# ---------------------------
# Tính BERTScore (PhoBERT)
# ---------------------------
from bert_score import score
import torch

def compute_bertscore(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = score(
        [candidate],
        [reference],
        lang="vi",          # QUAN TRỌNG
        model_type=None,    # KHÔNG ghi vinai/phobert-base
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }



# ---------------------------
# Hàm tổng hợp
# ---------------------------
def evaluate_summary(candidate, reference):
    candidate_proc = preprocess_vi(candidate)
    reference_proc = preprocess_vi(reference)

    candidate_text = " ".join(candidate_proc)
    reference_text = " ".join(reference_proc)

    results = {}
    results.update(compute_rouge(candidate_text, reference_text))
    results.update(compute_bertscore(candidate_text, reference_text))

    return results


import pandas as pd
import numpy as np

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest_sum.xlsx")
# Danh sách cột thang đo
metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "bertscore_precision", "bertscore_recall", "bertscore_f1"
]

# Nếu chưa có cột thì tạo
for col in metric_cols:
    if col not in df.columns:
        df[col] = np.nan

# =========================
# TÍNH ĐIỂM TỪNG ROW
# =========================
for idx, row in df.iterrows():
    content = row.get("content", "")
    summary = row.get("sum", "")

    # Bỏ qua nếu thiếu dữ liệu
    if not isinstance(content, str) or not isinstance(summary, str):
        continue
    if content.strip() == "" or summary.strip() == "":
        continue

    try:
        scores = evaluate_summary(summary, content)

        for k, v in scores.items():
            df.at[idx, k] = float(v)

    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue
    
df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest_score.xlsx", index=False)


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [2]:
# ============================================
# LOAD MODEL ĐÃ TRAIN
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import pandas as pd 

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest.xlsx")

MODEL_PATH = "../../Model_DG/vit5_grade_summary"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

# ============================================
# HÀM TÓM TẮT 1 VĂN BẢN
# ============================================

def summarize_text(content, grade,
                   max_input_len=512,
                   max_target_len=256):

    prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=max_target_len,
            num_beams=5,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return summary


# ============================================
# TEST VỚI 1 VĂN BẢN BẤT KỲ
# ============================================

grade = 5

df["sum"] = ""  

for i in range(len(df)):
    text = df.iloc[i]["content"]
    summary = summarize_text(text, grade)
    df.iloc[i, df.columns.get_loc("sum")] = summary

df.to_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest_sum.xlsx")


In [4]:
import pandas as pd 

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\diengiaitest_score.xlsx")
df.describe()

,Unnamed: 0,summary,grade,rouge1_precision,rouge1_recall,rouge1_f1,rougeL_precision,rougeL_recall,rougeL_f1,bertscore_precision,bertscore_recall,bertscore_f1
count,5.000000,0.0,5.000000,5.0,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,2.000000,NaN,3.000000,1.0,0.552875,0.704454,0.906374,0.502241,0.639062,0.874836,0.867130,0.870922
std,1.581139,NaN,1.581139,0.0,0.135036,0.109772,0.065762,0.139124,0.118406,0.021327,0.033938,0.027654
min,0.000000,NaN,1.000000,1.0,0.393293,0.564551,0.828571,0.381098,0.547046,0.837162,0.806988,0.821799
25%,1.000000,NaN,2.000000,1.0,0.480149,0.648785,0.844961,0.405707,0.548198,0.878655,0.877054,0.878797
50%,2.000000,NaN,3.000000,1.0,0.561224,0.718954,0.927083,0.465015,0.595705,0.884638,0.878940,0.882077
75%,3.000000,NaN,4.000000,1.0,0.572565,0.728192,0.962264,0.530815,0.675095,0.886568,0.883638,0.884138
max,4.000000,NaN,5.000000,1.0,0.757143,0.861789,0.968992,0.728571,0.829268,0.887158,0.889030,0.887797


In [16]:
import pandas as pd 

df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\Score model\Book1_score.xlsx")
df.describe()

,grade,rouge1_precision,rouge1_recall,rouge1_f1,rougeL_precision,rougeL_recall,rougeL_f1,bertscore_precision,bertscore_recall,bertscore_f1
count,2000.0,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,5.0,0.996919,0.347841,0.482581,0.905997,0.308463,0.429889,0.873448,0.865801,0.869564
std,0.0,0.012236,0.201106,0.218888,0.069523,0.171599,0.186263,0.027171,0.034485,0.030563
min,5.0,0.784173,0.028853,0.056087,0.565972,0.028853,0.056087,0.740772,0.691704,0.722325
25%,5.0,1.000000,0.174485,0.297126,0.868203,0.162896,0.277680,0.870202,0.864113,0.867360
50%,5.0,1.000000,0.328950,0.495052,0.920839,0.290840,0.435697,0.879970,0.875781,0.877694
75%,5.0,1.000000,0.492679,0.659733,0.958265,0.433918,0.577768,0.888052,0.884213,0.885779
max,5.0,1.000000,0.989011,0.953642,1.000000,0.900000,0.918367,0.934468,0.924712,0.929564
